# Import e test del modello di regressione logistica per il mutuo

Questo notebook:

- importa il modello esportato in `modello_mutuo.joblib`
- ispeziona **esplicitamente** la pipeline dopo il caricamento
- mostra le feature originali attese dal modello
- mostra le feature finali dopo il preprocessing
- mostra le categorie accettate per le feature categoriche
- collega le feature finali ai coefficienti della regressione logistica
- costruisce dati giocattolo coerenti con il formato atteso
- ottiene previsioni sia favorevoli sia sfavorevoli


In [6]:
#import pandas as pd
import numpy as np
from joblib import load


## 1. Caricamento del modello

Assunzione: il file `modello_mutuo.joblib` si trova nella stessa cartella del notebook.


In [7]:
model = load("modello_mutuo.joblib")
model



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\gianm\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\gianm\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\gianm\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\gianm\anaconda3\Lib\site-pack

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

## 2. Struttura della pipeline

Qui verifichiamo che l'oggetto caricato **non sia solo il classificatore**, ma una vera `Pipeline`
composta da:
- `preprocessor`
- `classifier`


In [3]:
print("Tipo oggetto caricato:", type(model).__name__)
print("\nPassi del pipeline principale:")
for nome, step in model.named_steps.items():
    print(f"- {nome}: {type(step).__name__}")


Tipo oggetto caricato: Pipeline

Passi del pipeline principale:
- preprocessor: ColumnTransformer
- classifier: LogisticRegression


In [4]:
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

print("Preprocessor:", type(preprocessor).__name__)
print("Classifier:", type(classifier).__name__)


Preprocessor: ColumnTransformer
Classifier: LogisticRegression


## 3. Feature originali attese dal modello

Queste sono le colonne che la pipeline si aspetta **prima** di applicare imputazione, scaling e one-hot encoding.


In [5]:
feature_originali = list(model.feature_names_in_)
feature_originali


['Gender',
 'Married',
 'Dependents',
 'Education',
 'Self_Employed',
 'ApplicantIncome',
 'CoapplicantIncome',
 'LoanAmount',
 'Loan_Amount_Term',
 'Credit_History',
 'Property_Area']

In [6]:
for i, f in enumerate(feature_originali, start=1):
    print(f"{i}. {f}")


1. Gender
2. Married
3. Dependents
4. Education
5. Self_Employed
6. ApplicantIncome
7. CoapplicantIncome
8. LoanAmount
9. Loan_Amount_Term
10. Credit_History
11. Property_Area


## 4. Trasformazioni applicate dal `preprocessor`

Il `preprocessor` è un `ColumnTransformer`: un ramo gestisce le feature numeriche,
un altro le feature categoriche.


In [7]:
transformers = preprocessor.transformers_

for nome, trasf, cols in transformers:
    print(f"\nTrasformatore: {nome}")
    print("Tipo:", type(trasf).__name__)
    print("Colonne:", list(cols))



Trasformatore: num
Tipo: Pipeline
Colonne: ['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

Trasformatore: cat
Tipo: Pipeline
Colonne: ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']


In [8]:
numeric_features = list(preprocessor.transformers_[0][2])
categorical_features = list(preprocessor.transformers_[1][2])

print("Feature numeriche:")
print(numeric_features)

print("\nFeature categoriche:")
print(categorical_features)


Feature numeriche:
['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

Feature categoriche:
['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']


## 5. Feature finali dopo il preprocessing

Questo è il punto più importante: il classificatore **non lavora più sulle feature originali**,
ma sulle feature trasformate prodotte dal `preprocessor`.


In [9]:
feature_finali = list(preprocessor.get_feature_names_out())
feature_finali


['num__Dependents',
 'num__ApplicantIncome',
 'num__CoapplicantIncome',
 'num__LoanAmount',
 'num__Loan_Amount_Term',
 'num__Credit_History',
 'cat__Gender_Female',
 'cat__Gender_Male',
 'cat__Married_No',
 'cat__Married_Yes',
 'cat__Education_Graduate',
 'cat__Education_Not Graduate',
 'cat__Self_Employed_No',
 'cat__Self_Employed_Yes',
 'cat__Property_Area_Rural',
 'cat__Property_Area_Semiurban',
 'cat__Property_Area_Urban']

In [10]:
for i, f in enumerate(feature_finali, start=1):
    print(f"{i}. {f}")


1. num__Dependents
2. num__ApplicantIncome
3. num__CoapplicantIncome
4. num__LoanAmount
5. num__Loan_Amount_Term
6. num__Credit_History
7. cat__Gender_Female
8. cat__Gender_Male
9. cat__Married_No
10. cat__Married_Yes
11. cat__Education_Graduate
12. cat__Education_Not Graduate
13. cat__Self_Employed_No
14. cat__Self_Employed_Yes
15. cat__Property_Area_Rural
16. cat__Property_Area_Semiurban
17. cat__Property_Area_Urban


## 6. Valori accettati dal modello

Per le variabili categoriche possiamo ispezionare direttamente le categorie apprese dal `OneHotEncoder`.

Per le variabili numeriche **non esiste un insieme chiuso di valori ammessi**:
il modello accetta numeri coerenti con il significato della colonna.


In [11]:
cat_pipeline = preprocessor.named_transformers_["cat"]
onehot = cat_pipeline.named_steps["onehot"]

categorie_accettate = {
    colonna: list(categorie)
    for colonna, categorie in zip(categorical_features, onehot.categories_)
}

categorie_accettate


{'Gender': ['Female', 'Male'],
 'Married': ['No', 'Yes'],
 'Education': ['Graduate', 'Not Graduate'],
 'Self_Employed': ['No', 'Yes'],
 'Property_Area': ['Rural', 'Semiurban', 'Urban']}

In [13]:
print("Categorie accettate per le feature categoriche:\n")
for colonna, categorie in categorie_accettate.items():
    print(f"- {colonna}: {categorie}")


Categorie accettate per le feature categoriche:

- Gender: ['Female', 'Male']
- Married: ['No', 'Yes']
- Education: ['Graduate', 'Not Graduate']
- Self_Employed: ['No', 'Yes']
- Property_Area: ['Rural', 'Semiurban', 'Urban']


In [14]:
print("Per le feature numeriche il modello si aspetta valori numerici:")
for colonna in numeric_features:
    print(f"- {colonna}: valore numerico")


Per le feature numeriche il modello si aspetta valori numerici:
- Dependents: valore numerico
- ApplicantIncome: valore numerico
- CoapplicantIncome: valore numerico
- LoanAmount: valore numerico
- Loan_Amount_Term: valore numerico
- Credit_History: valore numerico


## 7. Coefficienti del classificatore collegati alle feature finali

Qui colleghiamo ogni coefficiente della regressione logistica alla feature trasformata corrispondente.

Attenzione: il segno e il valore del coefficiente vanno interpretati con cautela,
perché alcune feature numeriche sono state standardizzate e quelle categoriche sono state espanse in dummy.


In [15]:
coefficienti = classifier.coef_[0]

df_coef = pd.DataFrame({
    "feature_finale": feature_finali,
    "coefficiente": coefficienti
})

df_coef


,feature_finale,coefficiente
0,num__Dependents,0.109813
1,num__ApplicantIncome,0.003787
2,num__CoapplicantIncome,-0.142085
3,num__LoanAmount,-0.043301
4,num__Loan_Amount_Term,-0.001388
5,num__Credit_History,1.235438
6,cat__Gender_Female,0.080516
7,cat__Gender_Male,-0.080749
8,cat__Married_No,-0.261892
9,cat__Married_Yes,0.261659


In [16]:
df_coef.sort_values("coefficiente", ascending=False)


,feature_finale,coefficiente
5,num__Credit_History,1.235438
15,cat__Property_Area_Semiurban,0.447011
9,cat__Married_Yes,0.261659
10,cat__Education_Graduate,0.199070
12,cat__Self_Employed_No,0.113043
0,num__Dependents,0.109813
6,cat__Gender_Female,0.080516
1,num__ApplicantIncome,0.003787
4,num__Loan_Amount_Term,-0.001388
3,num__LoanAmount,-0.043301


In [ ]:
df_coef.sort_values("coefficiente", ascending=True)


## 8. Dati giocattolo coerenti con il formato atteso

Costruiamo alcuni casi manuali.

Nota importante:
- i nomi delle colonne devono essere **esattamente** quelli attesi dal modello;
- le categorie testuali devono appartenere a quelle viste sopra;
- i campi numerici devono essere passati come numeri.


In [17]:
nuovi_clienti = pd.DataFrame([
    # Caso 1: profilo probabilmente favorevole
    {
        "Gender": "Male",
        "Married": "Yes",
        "Dependents": 0,
        "Education": "Graduate",
        "Self_Employed": "No",
        "ApplicantIncome": 9000,
        "CoapplicantIncome": 2500,
        "LoanAmount": 120,
        "Loan_Amount_Term": 360,
        "Credit_History": 1.0,
        "Property_Area": "Urban"
    },
    # Caso 2: profilo probabilmente sfavorevole
    {
        "Gender": "Female",
        "Married": "No",
        "Dependents": 3,
        "Education": "Not Graduate",
        "Self_Employed": "Yes",
        "ApplicantIncome": 1800,
        "CoapplicantIncome": 0,
        "LoanAmount": 220,
        "Loan_Amount_Term": 360,
        "Credit_History": 0.0,
        "Property_Area": "Rural"
    },
    # Caso 3: profilo intermedio
    {
        "Gender": "Male",
        "Married": "Yes",
        "Dependents": 1,
        "Education": "Graduate",
        "Self_Employed": "No",
        "ApplicantIncome": 5200,
        "CoapplicantIncome": 1800,
        "LoanAmount": 130,
        "Loan_Amount_Term": 360,
        "Credit_History": 1.0,
        "Property_Area": "Semiurban"
    }
])

nuovi_clienti


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area
0,Male,Yes,0,Graduate,No,9000,2500,120,360,1.0,Urban
1,Female,No,3,Not Graduate,Yes,1800,0,220,360,0.0,Rural
2,Male,Yes,1,Graduate,No,5200,1800,130,360,1.0,Semiurban


## 9. Previsioni del modello


In [18]:
predizioni = model.predict(nuovi_clienti)
probabilita = model.predict_proba(nuovi_clienti)[:, 1]

output = nuovi_clienti.copy()
output["Predizione_binaria"] = predizioni
output["Esito"] = output["Predizione_binaria"].map({1: "APPROVATO", 0: "RIFIUTATO"})
output["Prob_approvazione"] = np.round(probabilita, 3)

output


,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Predizione_binaria,Esito,Prob_approvazione
0,Male,Yes,0,Graduate,No,9000,2500,120,360,1.0,Urban,1,APPROVATO,0.784
1,Female,No,3,Not Graduate,Yes,1800,0,220,360,0.0,Rural,0,RIFIUTATO,0.046
2,Male,Yes,1,Graduate,No,5200,1800,130,360,1.0,Semiurban,1,APPROVATO,0.884


## 10. Controllo rapido: il modello restituisce sia approvati sia rifiutati?


In [ ]:
output[["Esito", "Prob_approvazione"]]


In [ ]:
print(output["Esito"].value_counts())


## 11. Funzione riutilizzabile per un singolo caso


In [ ]:
def predici_mutuo(dati_cliente: dict):
    nuovo = pd.DataFrame([dati_cliente])
    classe = model.predict(nuovo)[0]
    probabilita = model.predict_proba(nuovo)[0, 1]
    return {
        "predizione_binaria": int(classe),
        "esito": "APPROVATO" if classe == 1 else "RIFIUTATO",
        "probabilita_approvazione": round(float(probabilita), 3)
    }


In [ ]:
esempio_singolo = {
    "Gender": "Female",
    "Married": "Yes",
    "Dependents": 0,
    "Education": "Graduate",
    "Self_Employed": "No",
    "ApplicantIncome": 5200,
    "CoapplicantIncome": 1800,
    "LoanAmount": 130,
    "Loan_Amount_Term": 360,
    "Credit_History": 1.0,
    "Property_Area": "Urban"
}

predici_mutuo(esempio_singolo)


## Osservazione finale

Se in esecuzione tutti i casi risultassero approvati oppure tutti rifiutati, non sarebbe un errore del notebook:
vorrebbe dire solo che questi input manuali non stanno abbastanza vicino a entrambi i lati della frontiera decisionale del modello.

In quel caso basta modificare soprattutto:
- `Credit_History`
- `ApplicantIncome`
- `LoanAmount`
- `Property_Area`

per ottenere casi più contrastati.
